In [ ]:
# imports
import os
import re
import pandas as pd
from pathlib import Path
import hashlib
import ast

### Normalize column names

In [ ]:
target_columns = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]


def eldiario_image_filenames(image_urls):
    if pd.isna(image_urls) or not str(image_urls).strip():
        return ""
    urls = [u.strip() for u in str(image_urls).split("|") if u.strip()]
    return "|".join(hashlib.md5(u.encode("utf-8")).hexdigest() + ".jpg" for u in urls)


def elmundo_image_filenames(image_urls):
    if pd.isna(image_urls) or not str(image_urls).strip():
        return ""
    urls = [u.strip() for u in str(image_urls).split("|") if u.strip()]
    return "|".join(hashlib.md5(u.encode("utf-8")).hexdigest() + ".jpg" for u in urls)


raw_dataframes = {}

# eldiario: url -> article_url, published -> date, image_urls -> image_url
df_eldiario = pd.read_csv("eldiario_ukraine_russia_with_images.csv", low_memory=False)
df_eldiario["image_filename"] = df_eldiario["image_urls"].apply(eldiario_image_filenames)
df_eldiario = df_eldiario.rename(columns={
    "url": "article_url",
    "published": "date",
    "image_urls": "image_url",
})
raw_dataframes["eldiario"] = df_eldiario[target_columns]
print("eldiario reshaped:", len(raw_dataframes["eldiario"]), "articles")

# elmundo: url -> article_url, section -> category, published_time -> date, image_urls -> image_url
df_elmundo = pd.read_csv("elmundo_ukraine_russia.csv", low_memory=False)
df_elmundo["image_filename"] = df_elmundo["image_urls"].apply(elmundo_image_filenames)
df_elmundo = df_elmundo.rename(columns={
    "url": "article_url",
    "section": "category",
    "published_time": "date",
    "image_urls": "image_url",
})
raw_dataframes["elmundo"] = df_elmundo[target_columns]
print("elmundo reshaped:", len(raw_dataframes["elmundo"]), "articles")

In [ ]:
data_analysis_dir = "."
df_eldiario.to_csv(f"{data_analysis_dir}\\eldiario_full_data.csv", index=False, encoding="utf-8-sig")
print("eldiario_full_data.csv saved:", len(df_eldiario), "articles")
df_elmundo.to_csv(f"{data_analysis_dir}\\elmundo_full_data.csv", index=False, encoding="utf-8-sig")
print("elmundo_full_data.csv saved:", len(df_elmundo), "articles")

In [ ]:
import hashlib
import re
import pandas as pd

target_columns = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]
data_analysis_dir = "."


def spiegel_image_filenames(image_urls):
    # image_urls here is a python-list-repr string, e.g. "['https://...', 'https://...']"
    if pd.isna(image_urls) or not str(image_urls).strip():
        return ""
    try:
        urls = ast.literal_eval(image_urls)
    except (ValueError, SyntaxError):
        return ""
    return "|".join(hashlib.md5(u.encode("utf-8")).hexdigest() + ".jpg" for u in urls if u)


def welt_image_filenames(image_urls):
    if pd.isna(image_urls) or not str(image_urls).strip():
        return ""
    urls = [u.strip() for u in str(image_urls).split("|") if u.strip()]
    ids = []
    for u in urls:
        parts = u.split("/")
        if len(parts) >= 5:
            image_id = parts[4]  # https: / '' / images.welt.de / {1} / {image_id} / ...
            if image_id not in ids:
                ids.append(image_id)
    return "|".join(f"{i}.jpg" for i in ids)


# --- welt ---
df_welt = pd.read_csv(
    "welt_ukraine_russia_with_metadata.csv",
    low_memory=False,
)
df_welt["image_filename"] = df_welt["image_urls"].apply(welt_image_filenames)
df_welt = df_welt.rename(columns={
    "url": "article_url",
    "published": "date",
    "image_urls": "image_url",
})
df_welt = df_welt[target_columns]
df_welt.to_csv(f"{data_analysis_dir}\\welt_full_data.csv", index=False, encoding="utf-8-sig")
print("welt_full_data.csv saved:", len(df_welt), "articles")

In [ ]:
# --- spiegel ---
df_spiegel = pd.read_csv(
    "spiegel_clean_article_images_filtered.csv",
    low_memory=False,
)
before_drop = len(df_spiegel)
df_spiegel = df_spiegel[df_spiegel["article_text"].notna()].copy()
print("spiegel - dropped", before_drop - len(df_spiegel), "articles with no text (video-only pages)")

df_spiegel["image_filename"] = df_spiegel["image_urls"].apply(spiegel_image_filenames)
df_spiegel = df_spiegel.rename(columns={
    "section": "category",
    "published_time": "date",
    "article_text": "text",
    "image_urls": "image_url",
})
df_spiegel = df_spiegel[target_columns]
df_spiegel.to_csv(f"{data_analysis_dir}\\spiegel_full_data.csv", index=False, encoding="utf-8-sig")
print("spiegel_full_data.csv saved:", len(df_spiegel), "articles")

In [ ]:
data_analysis_dir = "."
target_columns = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]

df_polityka = pd.read_csv(
    "polityka_ukraine_russia_with_images.csv",
    low_memory=False,
)
index_df = pd.read_csv(
    "polityka_images_index.csv",
    low_memory=False,
)
url_to_filename = dict(zip(index_df["image_url"], index_df["filename"]))


def polityka_image_filenames(image_url):
    if pd.isna(image_url) or not str(image_url).strip():
        return ""
    urls = [u.strip() for u in str(image_url).split("|") if u.strip()]
    filenames = [url_to_filename[u] for u in urls if u in url_to_filename]
    return "|".join(filenames)


df_polityka["image_filename"] = df_polityka["image_url"].apply(polityka_image_filenames)
df_polityka = df_polityka.rename(columns={"url": "article_url", "publish_date": "date"})
df_polityka = df_polityka[target_columns]
df_polityka.to_csv(f"{data_analysis_dir}\\polityka_full_data.csv", index=False, encoding="utf-8-sig")
print("polityka_full_data.csv saved:", len(df_polityka), "articles")

In [ ]:
# reshape rp data: rename columns, build image_filename from image_urls, save full_data csv
data_analysis_dir = "."
target_columns = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]

rp_images_dir = "."
unique_df = pd.read_csv("rp_unique_images.csv", low_memory=False)
url_to_index = {url: i for i, url in enumerate(unique_df["image_url"])}

def rp_image_filenames(image_urls):
    if pd.isna(image_urls) or not str(image_urls).strip():
        return ""
    try:
        urls = ast.literal_eval(image_urls)
    except (ValueError, SyntaxError):
        return ""
    filenames = []
    for u in urls:
        idx = url_to_index.get(u)
        if idx is None:
            continue
        for ext in (".jpg", ".png", ".webp"):
            candidate = f"{idx:07d}{ext}"
            if os.path.exists(os.path.join(rp_images_dir, candidate)):
                filenames.append(candidate)
                break
    return "|".join(filenames)

df_rp = pd.read_csv("rp_russia_ukraine_with_images.csv", low_memory=False)
df_rp["image_filename"] = df_rp["image_urls"].apply(rp_image_filenames)
df_rp = df_rp.rename(columns={"url": "article_url", "image_urls": "image_url"})
df_rp["date"] = pd.NA  # rp_russia_ukraine_with_images.csv has no date/published column at all
df_rp = df_rp[target_columns]
df_rp.to_csv(f"{data_analysis_dir}\\rp_full_data.csv", index=False, encoding="utf-8-sig")
print("rp_full_data.csv saved:", len(df_rp), "articles")

In [ ]:
# reshape telex data: combine image urls, extract date from url, build image_filename, save full_data csv
target_columns = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]
data_analysis_dir = "."

date_pattern = re.compile(r"/(\d{4})/(\d{2})/(\d{2})/")

def telex_extract_date(article_url):
    match = date_pattern.search(str(article_url))
    if not match:
        return pd.NA
    year, month, day = match.groups()
    return f"{year}-{month}-{day}"

def telex_combine_image_urls(og_image, body_images):
    urls = []
    if isinstance(og_image, str) and og_image.strip():
        urls.append(og_image.strip())
    if isinstance(body_images, str) and body_images.strip():
        try:
            parsed = ast.literal_eval(body_images)
        except (ValueError, SyntaxError):
            parsed = []
        for u in parsed:
            if u and u not in urls:
                urls.append(u)
    return "|".join(urls)

def telex_image_filenames(image_url):
    if pd.isna(image_url) or not str(image_url).strip():
        return ""
    urls = [u.strip() for u in str(image_url).split("|") if u.strip()]
    return "|".join(hashlib.md5(u.encode("utf-8")).hexdigest() + ".jpg" for u in urls)

df_telex = pd.read_csv("telex_ukraine_russia_with_images_cleaned.csv", low_memory=False)
df_telex["image_url"] = df_telex.apply(lambda row: telex_combine_image_urls(row["og_image"], row["body_images"]), axis=1)
df_telex["image_filename"] = df_telex["image_url"].apply(telex_image_filenames)
df_telex["date"] = df_telex["article_url"].apply(telex_extract_date)
df_telex = df_telex.rename(columns={"article_text": "text"})
df_telex = df_telex[target_columns]

In [ ]:
df_telex.to_csv(f"{data_analysis_dir}\\telex_full_data.csv", index=False, encoding="utf-8-sig")
print("telex_full_data.csv saved:", len(df_telex), "articles")

### New narrow filter

In [ ]:
# paths to the 8 raw keyword-filtered datasets and their image folders

sources = [
    {"name": "elmundo",  "language": "es", "csv": "elmundo_full_data.csv",  "images_folder": "."},
    {"name": "eldiario", "language": "es", "csv": "eldiario_full_data.csv", "images_folder": "."},
    {"name": "rp",       "language": "pl", "csv": "rp_full_data.csv",       "images_folder": "."},
    {"name": "polityka", "language": "pl", "csv": "polityka_full_data.csv", "images_folder": "."},
    {"name": "magyar",   "language": "hu", "csv": "magyar_full_data.csv",   "images_folder": "."},
    {"name": "telex",    "language": "hu", "csv": "telex_full_data.csv",    "images_folder": "."},
    {"name": "spiegel",  "language": "de", "csv": "spiegel_full_data.csv",  "images_folder": "."},
    {"name": "welt",     "language": "de", "csv": "welt_full_data.csv",     "images_folder": "."},
]

In [ ]:
# keyword patterns per language, matched against the title

russia_keywords = {
    "de": re.compile(
        r"\b("
        r"russland\w*|russisch\w*|kreml\w*|putin\w*|moskau\w*|"
        r"lawrow\w*|lavrov\w*|peskow\w*|peskov\w*|schoigu\w*|shoigu\w*|"
        r"gerassimow\w*|gerasimow\w*|medwedew\w*|sacharowa\w*|mischustin\w*|"
        r"gazprom\w*|rosneft\w*|rosatom\w*|transneft\w*|lukoil\w*|"
        r"donezk\w*|luhansk\w*|krim\w*|crimea\w*"
        r")\b",
        re.IGNORECASE,
    ),
    "hu": re.compile(
        r"\b("
        r"oroszorsz[aá]g\w*|orosz\w*|kreml\w*|putyin\w*|moszkv\w*|"
        r"lavrov\w*|peszkov\w*|peskov\w*|sojgu\w*|szojgu\w*|shoigu\w*|"
        r"geraszimov\w*|gerasimov\w*|medvegyev\w*|medvedev\w*|zaharova\w*|zakharova\w*|misustyin\w*|mishustin\w*|"
        r"gazprom\w*|rosznyefty\w*|rosneft\w*|roszatom\w*|rosatom\w*|transznyefty\w*|transneft\w*|lukoil\w*|"
        r"donyeck\w*|donetsk\w*|luhanszk\w*|kr[ií]m\w*"
        r")\b",
        re.IGNORECASE,
    ),
    "pl": re.compile(
        r"\b("
        r"rosj\w*|rosyjsk\w*|kreml\w*|putin\w*|moskw\w*|"
        r"ławrow\w*|lawrow\w*|pieskow\w*|szojgu\w*|szoigu\w*|shoigu\w*|"
        r"gierasimow\w*|miedwiediew\w*|zacharowa\w*|miszustin\w*|"
        r"gazprom\w*|rosnieft\w*|rosatom\w*|transnieft\w*|łukoil\w*|lukoil\w*|"
        r"donieck\w*|ługańsk\w*|luhansk\w*|krym\w*"
        r")\b",
        re.IGNORECASE,
    ),
    "es": re.compile(
        r"\b("
        r"rusia\w*|rus[oa]s?|kremlin\w*|putin\w*|mosc[uú]\w*|"
        r"lavrov\w*|peskov\w*|shoigu\w*|choigu\w*|gerasimov\w*|"
        r"medv[eé]dev\w*|zaj[aá]rova\w*|zakharova\w*|mishustin\w*|mishustín\w*|"
        r"gazprom\w*|rosneft\w*|rosatom\w*|transneft\w*|lukoil\w*|"
        r"donetsk\w*|lugansk\w*|crimea\w*"
        r")\b",
        re.IGNORECASE,
    ),
}
def title_mentions_russia(title, language):
    pattern = russia_keywords[language]
    if not isinstance(title, str):
        return False
    return bool(pattern.search(title))


dataframes = {}

for source in sources:
    df = pd.read_csv(source["csv"], low_memory=False)
    n_before = len(df)

    matched = df["title"].apply(lambda t: title_mentions_russia(t, source["language"]))
    df = df[matched].copy()

    dataframes[source["name"]] = df
    print(source["name"], "-", n_before, "->", len(df), "articles with Russia in the title")

In [ ]:
# rp: after keyword filtering, deduplicate image_filename by underlying photo (same id, different crop size)

rp_images_dir = "."
unique_df = pd.read_csv("rp_unique_images.csv", low_memory=False)
url_to_index = {url: i for i, url in enumerate(unique_df["image_url"])}

def rp_extract_image_id(url):
    match = re.search(r"/image/free/([a-f0-9]+)/", str(url))
    return match.group(1) if match else url

def rp_extract_resolution(url):
    match = re.search(r"resize:fill:(\d+):(\d+)", str(url))
    if match:
        return int(match.group(1)) * int(match.group(2))
    return 0

def rp_dedupe_image_filenames(image_url):
    if pd.isna(image_url) or not str(image_url).strip():
        return ""
    try:
        urls = ast.literal_eval(image_url)
    except (ValueError, SyntaxError):
        return ""

    best_url_by_id = {}
    for u in urls:
        image_id = rp_extract_image_id(u)
        if image_id not in best_url_by_id or rp_extract_resolution(u) > rp_extract_resolution(best_url_by_id[image_id]):
            best_url_by_id[image_id] = u

    filenames = []
    for u in best_url_by_id.values():
        idx = url_to_index.get(u)
        if idx is None:
            continue
        for ext in (".jpg", ".png", ".webp"):
            candidate = f"{idx:07d}{ext}"
            if os.path.exists(os.path.join(rp_images_dir, candidate)):
                filenames.append(candidate)
                break
    return "|".join(filenames)

count_before = dataframes["rp"]["image_filename"].fillna("").apply(lambda f: len([x for x in str(f).split("|") if x.strip()])).sum()
dataframes["rp"]["image_filename"] = dataframes["rp"]["image_url"].apply(rp_dedupe_image_filenames)
count_after = dataframes["rp"]["image_filename"].fillna("").apply(lambda f: len([x for x in str(f).split("|") if x.strip()])).sum()

print("rp - total image references:", count_before, "->", count_after, "after deduplicating by underlying photo")

### task 2: check missing images, using the already-filtered dataframes from the keyword step

In [ ]:
# check missing images: verify every image_filename referenced in the filtered data exists on disk
missing_images = {}
for source in sources:
    name = source["name"]
    df = dataframes[name]
    image_dir = Path(source["images_folder"])
    if not image_dir.exists():
        print(name, "- images folder does not exist:", image_dir)
        continue

    referenced = set()
    for filenames in df["image_filename"].fillna(""):
        for filename in str(filenames).split("|"):
            filename = filename.strip()
            if filename:
                referenced.add(filename)

    missing = {f for f in referenced if not (image_dir / f).exists()}
    missing_images[name] = missing

    no_image_at_all = df["image_filename"].fillna("").apply(lambda f: not str(f).strip()).sum()
    has_missing = df["image_filename"].fillna("").apply(
        lambda f: any(name in missing for name in str(f).split("|") if name.strip())
    ).sum()

    print(name, "-", len(referenced), "filenames referenced,", len(missing), "missing on disk,",
          no_image_at_all, "articles with no image at all,", has_missing, "articles with at least one missing image")

In [ ]:
# print raw image_url values for 10 "no image" articles per source, for manual verification
for name in ["rp", "telex"]:
    df = dataframes[name]
    no_image_mask = df["image_filename"].fillna("").apply(lambda f: not str(f).strip())
    sample = df[no_image_mask][["article_url", "image_url"]].head(10)
    print(name)
    for _, row in sample.iterrows():
        print(" article_url:", row["article_url"])
        print(" image_url:", repr(row["image_url"]))
        print()

In [ ]:
# check whether "no image" articles genuinely had no image urls, or had urls that failed to resolve
def count_source_urls(image_url):
    if pd.isna(image_url) or not str(image_url).strip():
        return 0
    text = str(image_url).strip()
    if text.startswith("["):
        try:
            return len(ast.literal_eval(text))
        except (ValueError, SyntaxError):
            return 0
    return len(text.split("|"))

for name in ["rp", "telex"]:
    df = dataframes[name]
    no_image_mask = df["image_filename"].fillna("").apply(lambda f: not str(f).strip())
    no_image_df = df[no_image_mask]
    url_counts = no_image_df["image_url"].apply(count_source_urls)
    print(name, "- no-image articles:", len(no_image_df),
          "| genuinely had zero image urls:", (url_counts == 0).sum(),
          "| had urls but none resolved to a file:", (url_counts > 0).sum())

In [ ]:
# find the source urls behind missing spiegel image filenames, for manual inspection
df_spiegel_raw = pd.read_csv("spiegel_clean_article_images_filtered.csv", low_memory=False)

hash_to_url = {}
for image_urls in df_spiegel_raw["image_urls"].dropna():
    try:
        urls = ast.literal_eval(image_urls)
    except (ValueError, SyntaxError):
        continue
    for u in urls:
        hash_to_url[hashlib.md5(u.encode("utf-8")).hexdigest() + ".jpg"] = u

sample_missing = list(missing_images["spiegel"])[:10]
for filename in sample_missing:
    print(filename, "->", hash_to_url.get(filename, "url not found"))

### download mising images for spiegel

In [ ]:
# download missing spiegel images (same md5 + real-extension scheme as the original script), into a separate new folder
import ast
import hashlib
import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

input_file = "spiegel_clean_article_images_filtered.csv"
existing_dir = "."
new_dir = "."
failed_file = "spiegel_failed_unique_downloads_batch2.csv"

max_workers = 10
timeout = 30

os.makedirs(new_dir, exist_ok=True)

headers = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )
}

def make_filename(url):
    url_hash = hashlib.md5(url.encode()).hexdigest()
    lower = url.lower()
    if ".jpg" in lower:
        ext = ".jpg"
    elif ".jpeg" in lower:
        ext = ".jpeg"
    elif ".png" in lower:
        ext = ".png"
    elif ".webp" in lower:
        ext = ".webp"
    elif ".avif" in lower:
        ext = ".avif"
    else:
        ext = ".jpg"
    return url_hash + ext

# collect all unique image urls from the full spiegel corpus, same as in the original download notebook
df = pd.read_csv(input_file)

all_images = []
for image_urls in df["image_urls"].dropna():
    try:
        urls = ast.literal_eval(image_urls)
    except (ValueError, SyntaxError):
        continue
    all_images.extend(urls)

unique_images = sorted(set(all_images))
print("all unique images:", len(unique_images))

# keep only images not already present in the existing folder
to_download = [u for u in unique_images if not os.path.exists(os.path.join(existing_dir, make_filename(u)))]
print("still missing:", len(to_download))

def download_one(url):
    filename = make_filename(url)
    filepath = os.path.join(new_dir, filename)
    if os.path.exists(filepath):
        return ("skipped", url, None)
    try:
        response = requests.get(url, headers=headers, timeout=timeout, stream=True)
        if response.status_code != 200:
            return ("failed", url, f"status {response.status_code}")
        with open(filepath, "wb") as f:
            for chunk in response.iter_content(8192):
                f.write(chunk)
        return ("downloaded", url, None)
    except Exception as e:
        return ("failed", url, str(e))

results = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(download_one, url): url for url in to_download}
    for i, future in enumerate(as_completed(futures), start=1):
        results.append(future.result())
        if i % 500 == 0:
            print(i, "/", len(to_download), "processed")

downloaded = sum(1 for status, _, _ in results if status == "downloaded")
failed = [(url, error) for status, url, error in results if status == "failed"]

print("downloaded:", downloaded)
print("failed:", len(failed))

failed_df = pd.DataFrame(failed, columns=["image_url", "error"])
failed_df.to_csv(failed_file, index=False, encoding="utf-8-sig")
print("failed log saved:", failed_file)

In [ ]:
# check the newly downloaded spiegel batch2 images for junk, before merging into the main folder
import os
import re
import ast
import hashlib
import shutil
from pathlib import Path
import pandas as pd

final_dir = "."
junk_size_kb_threshold = 5
junk_reuse_threshold = 4

name = "spiegel"
source = {
    "name": "spiegel",
    "language": "de",
    "csv": "spiegel_full_data.csv",
    "images_folder": ".",
}

de_pattern = re.compile(
    r"\b("
    r"russland\w*|russisch\w*|kreml\w*|putin\w*|moskau\w*|"
    r"lawrow\w*|lavrov\w*|peskow\w*|peskov\w*|schoigu\w*|shoigu\w*|"
    r"gerassimow\w*|gerasimow\w*|medwedew\w*|sacharowa\w*|mischustin\w*|"
    r"gazprom\w*|rosneft\w*|rosatom\w*|transneft\w*|lukoil\w*|"
    r"donezk\w*|luhansk\w*|krim\w*|crimea\w*"
    r")\b",
    re.IGNORECASE,
)

def title_mentions_russia(title):
    if not isinstance(title, str):
        return False
    return bool(de_pattern.search(title))

# rebuild dataframes["spiegel"]: same keyword filter as the original pipeline, applied to spiegel_full_data.csv
df_raw = pd.read_csv(source["csv"], low_memory=False)
matched = df_raw["title"].apply(title_mentions_russia)
df = df_raw[matched].copy()
dataframes = {"spiegel": df}

missing_images = {}
junk_filenames = {}
clean_dataframes = {}

print("spiegel -", len(df_raw), "->", len(df), "articles with Russia in the title (rebuilt after restart)")

In [ ]:
# check the newly downloaded spiegel batch2 images for junk, before merging into the main folder
batch2_dir = Path(".")
df = dataframes[name]

usage_count = {}
for filenames in df["image_filename"].fillna(""):
    for filename in str(filenames).split("|"):
        filename = filename.strip()
        if filename:
            usage_count[filename] = usage_count.get(filename, 0) + 1

batch2_files = [f.name for f in batch2_dir.glob("*") if f.is_file()]
print("batch2 files downloaded:", len(batch2_files))

small_files = [f for f in batch2_files if (batch2_dir / f).stat().st_size / 1024 < junk_size_kb_threshold]
reused_files = [f for f in batch2_files if usage_count.get(f, 0) >= junk_reuse_threshold]

print("under", junk_size_kb_threshold, "kb:", len(small_files))
print("reused across", junk_reuse_threshold, "+ articles:", len(reused_files))
print("overlap (both):", len(set(small_files) & set(reused_files)))

In [ ]:
# merge batch2 downloads into the main spiegel image folder (skip files already present)
main_dir = Path(source["images_folder"])

merged = 0
for filename in batch2_files:
    src = batch2_dir / filename
    dst = main_dir / filename
    if not dst.exists():
        shutil.copy2(src, dst)
        merged += 1

print(merged, "files merged into", main_dir)

In [ ]:
# spiegel only: recompute missing/junk images now that batch2 is merged in
image_dir = Path(source["images_folder"])

referenced = set()
for filenames in df["image_filename"].fillna(""):
    for filename in str(filenames).split("|"):
        filename = filename.strip()
        if filename:
            referenced.add(filename)

missing_images[name] = {f for f in referenced if not (image_dir / f).exists()}
reused = {f for f, count in usage_count.items() if count >= junk_reuse_threshold}
small_files_disk = {f for f in usage_count if (image_dir / f).exists() and (image_dir / f).stat().st_size / 1024 < junk_size_kb_threshold}
junk_filenames[name] = reused | small_files_disk

print(name, "-", len(missing_images[name]), "missing,", len(junk_filenames[name]), "junk (recomputed after batch2 merge)")

In [ ]:
# spiegel only: redo steps 1-3 (dedupe articles, remove nan text, clean image_filename) on the updated data
normalized_text = df["text"].fillna("").str.strip().str.lower()
df = df[~normalized_text.duplicated()].copy()
df = df[df["text"].notna()].copy()

bad_filenames = missing_images[name] | junk_filenames[name]

def clean_filenames(filenames):
    kept = [f.strip() for f in str(filenames).split("|") if f.strip() and f.strip() not in bad_filenames]
    return "|".join(kept)

before_with_image = (df["image_filename"].fillna("").str.strip() != "").sum()
df["image_filename"] = df["image_filename"].fillna("").apply(clean_filenames)
after_with_image = (df["image_filename"].str.strip() != "").sum()

clean_dataframes[name] = df
print(name, "-", len(df), "articles after redoing steps 1-3")
print(name, "-", before_with_image, "->", after_with_image, "articles with at least one valid image")

In [ ]:
# spiegel only: sync spiegel_final_images - remove files no longer needed, copy the newly valid ones
df = clean_dataframes[name]
source_dir = Path(source["images_folder"])
dest_dir = Path(final_dir) / f"{name}_final_images"

needed_filenames = set()
for filenames in df["image_filename"].fillna(""):
    for filename in str(filenames).split("|"):
        filename = filename.strip()
        if filename:
            needed_filenames.add(filename)

removed = 0
for existing_file in dest_dir.glob("*"):
    if existing_file.name not in needed_filenames:
        existing_file.unlink()
        removed += 1

copied = 0
for filename in needed_filenames:
    dst_path = dest_dir / filename
    if not dst_path.exists():
        src_path = source_dir / filename
        if src_path.exists():
            shutil.copy2(src_path, dst_path)
            copied += 1

print(name, "-", len(needed_filenames), "needed,", removed, "old files removed,", copied, "new files copied")

In [ ]:
# how many of the newly downloaded batch2 images actually count as valid now (passed the junk filter)
recovered_valid_filenames = set(batch2_files) - missing_images[name] - junk_filenames[name]
print(len(batch2_files), "images downloaded in batch2,", len(recovered_valid_filenames), "of them count as valid (not junk)")

In [ ]:
# total valid image references at the row level (an article with 3 valid images counts as 3), before vs after this download round
def count_valid_refs(filenames, bad_set):
    return sum(1 for f in str(filenames).split("|") if f.strip() and f.strip() not in bad_set)

bad_now = missing_images[name] | junk_filenames[name]
total_valid_after = sum(count_valid_refs(f, bad_now) for f in dataframes[name]["image_filename"].fillna(""))

# "before" = same references, but treat every batch2 file as if it were still missing
bad_before = bad_now | set(batch2_files)
total_valid_before = sum(count_valid_refs(f, bad_before) for f in dataframes[name]["image_filename"].fillna(""))

print("total valid image references - before:", total_valid_before, "-> after:", total_valid_after,
      "(+", total_valid_after - total_valid_before, ")")

In [ ]:
# spiegel only: overwrite the final csv
output_path = os.path.join(final_dir, f"{name}_final_data.csv")
clean_dataframes[name].to_csv(output_path, index=False, encoding="utf-8-sig")
print(name, "-", output_path, "-", len(clean_dataframes[name]), "articles saved")

In [ ]:
# convert image_filename separator back to ";" for the freshly rewritten spiegel csv
csv_path = os.path.join(final_dir, "spiegel_final_data.csv")
df_check = pd.read_csv(csv_path, low_memory=False)

already_has_semicolon = df_check["image_filename"].fillna("").str.contains(";").sum()
if already_has_semicolon:
    print("WARNING:", already_has_semicolon, "rows already contain ';' — check before proceeding")
else:
    before_pipe_count = df_check["image_filename"].fillna("").str.count(r"\|").sum()
    df_check["image_filename"] = df_check["image_filename"].fillna("").str.replace("|", ";", regex=False)
    after_semicolon_count = df_check["image_filename"].str.count(";").sum()
    df_check.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print("spiegel -", len(df_check), "rows saved,", before_pipe_count, "'|' replaced with", after_semicolon_count, "';'")

### task 3: duplicate and junk images, using the filtered dataframes — a filename reused

In [ ]:
# check for junk images: reused across many articles (logos, banners, author photos) or too small (icons)
junk_size_kb_threshold = 5
junk_reuse_threshold = 4

junk_filenames = {}

for source in sources:
    name = source["name"]
    df = dataframes[name]
    image_dir = Path(source["images_folder"])
    if not image_dir.exists():
        print(name, "- images folder does not exist:", image_dir)
        continue

    usage_count = {}
    for filenames in df["image_filename"].fillna(""):
        for filename in str(filenames).split("|"):
            filename = filename.strip()
            if filename:
                usage_count[filename] = usage_count.get(filename, 0) + 1

    reused = {f for f, count in usage_count.items() if count >= junk_reuse_threshold}

    small_files = set()
    for filename in usage_count:
        path = image_dir / filename
        if path.exists() and path.stat().st_size / 1024 < junk_size_kb_threshold:
            small_files.add(filename)

    junk_filenames[name] = reused | small_files

    print(name, "-", len(reused), "reused across", junk_reuse_threshold, "+ articles,",
          len(small_files), "under", junk_size_kb_threshold, "kb")
    if reused:
        print("  most reused:", sorted(reused, key=lambda f: -usage_count[f])[:5])

In [ ]:
# check for duplicate articles: exact duplicate article_url, and duplicate text under different urls
duplicate_report = {}
for source in sources:
    name = source["name"]
    df = dataframes[name]

    url_duplicates = df["article_url"].duplicated().sum()
    normalized_text = df["text"].fillna("").str.strip().str.lower()
    text_duplicates = normalized_text.duplicated().sum()

    duplicate_report[name] = {"url_duplicates": url_duplicates, "text_duplicates": text_duplicates}
    print(name, "-", url_duplicates, "duplicate article_url,", text_duplicates, "duplicate text under different urls")

In [ ]:
# check whether eldiario articles with missing "text" have usable content in fallback columns
import pandas as pd

df_eldiario_raw = pd.read_csv("eldiario_ukraine_russia_with_images.csv", low_memory=False)

print("total rows:", len(df_eldiario_raw))
print("has_full_text value counts:")
print(df_eldiario_raw["has_full_text"].value_counts(dropna=False))

missing_text = df_eldiario_raw[df_eldiario_raw["text"].isna()]
print("\nrows with missing text:", len(missing_text))
print("of those, has_full_text values:")
print(missing_text["has_full_text"].value_counts(dropna=False))

print("\nof those, text_preview non-empty:", missing_text["text_preview"].notna().sum())
print("of those, combined_text non-empty:", missing_text["combined_text"].notna().sum())

print("\nsample text_preview for missing-text rows:")
for _, row in missing_text.head(5).iterrows():
    print(" title:", row["title"])
    print(" text_preview:", repr(row["text_preview"])[:200])
    print(" combined_text:", repr(row["combined_text"])[:200])
    print()

In [ ]:
# reshape eldiario data: rename columns, fill missing text from text_preview, build image_filename, save full_data csv
target_columns = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]
data_analysis_dir = "."

def eldiario_image_filenames(image_urls):
    if pd.isna(image_urls) or not str(image_urls).strip():
        return ""
    urls = [u.strip() for u in str(image_urls).split("|") if u.strip()]
    return "|".join(hashlib.md5(u.encode("utf-8")).hexdigest() + ".jpg" for u in urls)

df_eldiario = pd.read_csv("eldiario_ukraine_russia_with_images.csv", low_memory=False)
df_eldiario["image_filename"] = df_eldiario["image_urls"].apply(eldiario_image_filenames)

filled_count = df_eldiario["text"].isna().sum()
df_eldiario["text"] = df_eldiario["text"].fillna(df_eldiario["text_preview"])

df_eldiario = df_eldiario.rename(columns={"url": "article_url", "published": "date", "image_urls": "image_url"})
df_eldiario = df_eldiario[target_columns]
df_eldiario.to_csv(f"{data_analysis_dir}\\eldiario_full_data.csv", index=False, encoding="utf-8-sig")
print("eldiario_full_data.csv saved:", len(df_eldiario), "articles")
print("text filled from text_preview:", filled_count, "articles")

## Creating clean folder 

In [ ]:
# rp only: recompute missing/junk images on the new deduplicated image_filename set
name = "rp"
source = next(s for s in sources if s["name"] == name)
df = dataframes[name]
image_dir = Path(source["images_folder"])

referenced = set()
for filenames in df["image_filename"].fillna(""):
    for filename in str(filenames).split("|"):
        filename = filename.strip()
        if filename:
            referenced.add(filename)

missing_images[name] = {f for f in referenced if not (image_dir / f).exists()}

usage_count = {}
for filenames in df["image_filename"].fillna(""):
    for filename in str(filenames).split("|"):
        filename = filename.strip()
        if filename:
            usage_count[filename] = usage_count.get(filename, 0) + 1

reused = {f for f, count in usage_count.items() if count >= junk_reuse_threshold}
small_files = {f for f in usage_count if (image_dir / f).exists() and (image_dir / f).stat().st_size / 1024 < junk_size_kb_threshold}
junk_filenames[name] = reused | small_files

print(name, "-", len(missing_images[name]), "missing,", len(junk_filenames[name]), "junk (recomputed after dedup)")

In [ ]:
# rp only: redo steps 1-3 (dedupe articles, remove nan text, clean image_filename) on the new data
df = dataframes[name]

normalized_text = df["text"].fillna("").str.strip().str.lower()
df = df[~normalized_text.duplicated()].copy()

df = df[df["text"].notna()].copy()

bad_filenames = missing_images[name] | junk_filenames[name]

def clean_filenames(filenames):
    kept = [f.strip() for f in str(filenames).split("|") if f.strip() and f.strip() not in bad_filenames]
    return "|".join(kept)

df["image_filename"] = df["image_filename"].fillna("").apply(clean_filenames)

clean_dataframes[name] = df
print(name, "-", len(df), "articles after redoing steps 1-3")

In [ ]:
# rp only: sync rp_final_images - remove files no longer needed, keep/copy the rest
df = clean_dataframes[name]
source_dir = Path(source["images_folder"])
dest_dir = Path(final_dir) / f"{name}_final_images"

needed_filenames = set()
for filenames in df["image_filename"].fillna(""):
    for filename in str(filenames).split("|"):
        filename = filename.strip()
        if filename:
            needed_filenames.add(filename)

removed = 0
for existing_file in dest_dir.glob("*"):
    if existing_file.name not in needed_filenames:
        existing_file.unlink()
        removed += 1

copied = 0
for filename in needed_filenames:
    dst_path = dest_dir / filename
    if not dst_path.exists():
        src_path = source_dir / filename
        if src_path.exists():
            shutil.copy2(src_path, dst_path)
            copied += 1

print(name, "-", len(needed_filenames), "needed,", removed, "old files removed,", copied, "new files copied")

In [ ]:
# rp only: overwrite the final csv
output_path = os.path.join(final_dir, f"{name}_final_data.csv")
clean_dataframes[name].to_csv(output_path, index=False, encoding="utf-8-sig")
print(name, "-", output_path, "-", len(clean_dataframes[name]), "articles saved")

In [ ]:
final_dir = "."

for source in sources:
    name = source["name"]
    folder_path = os.path.join(final_dir, f"{name}_final_images")
    os.makedirs(folder_path, exist_ok=True)
    print(name, "-", folder_path)

In [ ]:
# step 1: remove duplicate articles (same text under different urls), keep the first occurrence
clean_dataframes = {}

for source in sources:
    name = source["name"]
    df = dataframes[name]

    normalized_text = df["text"].fillna("").str.strip().str.lower()
    before = len(df)
    df = df[~normalized_text.duplicated()].copy()

    clean_dataframes[name] = df
    print(name, "-", before, "->", len(df), "articles after removing duplicate text")

In [ ]:
# step 2: remove articles with no text at all (nan) - short-but-present texts are kept
for source in sources:
    name = source["name"]
    df = clean_dataframes[name]
    before = len(df)
    df = df[df["text"].notna()].copy()
    clean_dataframes[name] = df
    print(name, "-", before, "->", len(df), "articles after removing nan text")

In [ ]:
# step 3: remove missing-on-disk and junk filenames from image_filename, keep only valid images
for source in sources:
    name = source["name"]
    df = clean_dataframes[name]
    bad_filenames = missing_images.get(name, set()) | junk_filenames.get(name, set())

    def clean_filenames(filenames):
        kept = [f.strip() for f in str(filenames).split("|") if f.strip() and f.strip() not in bad_filenames]
        return "|".join(kept)

    before_with_image = (df["image_filename"].fillna("").str.strip() != "").sum()
    df["image_filename"] = df["image_filename"].fillna("").apply(clean_filenames)
    after_with_image = (df["image_filename"].str.strip() != "").sum()

    clean_dataframes[name] = df
    print(name, "-", before_with_image, "->", after_with_image, "articles with at least one valid image")

In [ ]:
# step 4: copy all valid images (referenced in clean_dataframes) into the final images folder per source
import shutil

final_dir = "."

for source in sources:
    name = source["name"]
    df = clean_dataframes[name]
    source_dir = Path(source["images_folder"])
    dest_dir = Path(final_dir) / f"{name}_final_images"

    needed_filenames = set()
    for filenames in df["image_filename"].fillna(""):
        for filename in str(filenames).split("|"):
            filename = filename.strip()
            if filename:
                needed_filenames.add(filename)

    copied = 0
    missing_at_copy = 0
    for filename in needed_filenames:
        src_path = source_dir / filename
        dst_path = dest_dir / filename
        if dst_path.exists():
            copied += 1
            continue
        if src_path.exists():
            shutil.copy2(src_path, dst_path)
            copied += 1
        else:
            missing_at_copy += 1

    print(name, "-", len(needed_filenames), "needed,", copied, "copied/already present,", missing_at_copy, "missing at copy time")

In [ ]:
# step 5: save final cleaned csv per source
final_dir = "."

for source in sources:
    name = source["name"]
    df = clean_dataframes[name]
    output_path = os.path.join(final_dir, f"{name}_final_data.csv")
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(name, "-", output_path, "-", len(df), "articles saved")

In [ ]:
# check that no filename already contains ";" (would signal a problem), then convert "|" to ";" in image_filename for all 8 final datasets
final_dir = "."
all_source_names = ["elmundo", "eldiario", "rp", "polityka", "magyar", "telex", "spiegel", "welt"]

for name in all_source_names:
    csv_path = os.path.join(final_dir, f"{name}_final_data.csv")
    df = pd.read_csv(csv_path, low_memory=False)

    already_has_semicolon = df["image_filename"].fillna("").str.contains(";").sum()
    if already_has_semicolon:
        print(name, "- WARNING:", already_has_semicolon, "rows already contain ';' in image_filename, skipping this file")
        continue

    before_pipe_count = df["image_filename"].fillna("").str.count(r"\|").sum()
    df["image_filename"] = df["image_filename"].fillna("").str.replace("|", ";", regex=False)
    after_semicolon_count = df["image_filename"].str.count(";").sum()

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(name, "-", len(df), "rows saved, ", before_pipe_count, "'|' replaced with", after_semicolon_count, "';'")

In [ ]:
final_dir = Path(".")

df_disk = pd.read_csv(final_dir / "spiegel_final_data.csv", low_memory=False)

uses_semicolon = df_disk["image_filename"].fillna("").str.contains(";").sum() > 0
uses_pipe = df_disk["image_filename"].fillna("").str.contains(r"\|", regex=True).sum() > 0
has_image = (df_disk["image_filename"].fillna("").str.strip() != "").sum()

print("rows on disk:", len(df_disk))
print("uses ';' separator:", uses_semicolon, "| uses '|' separator:", uses_pipe)
print("articles with at least one image reference:", has_image)
print("expected (from the redo): 13754 rows, 13516 with a valid image")

In [ ]:
import re

final_dir = Path(".")
all_source_names = ["elmundo", "eldiario", "rp", "polityka", "magyar", "telex", "spiegel", "welt"]

print(f"{'source':<10} {'csv_ok':>7} {'folder_ok':>10} {'rows':>7} {'nan_text':>9} {'dup_url':>8} {'sep':>5} {'with_img':>9} {'missing_img':>12} {'orphan_files':>13}")

for name in all_source_names:
    csv_path = final_dir / f"{name}_final_data.csv"
    image_dir = final_dir / f"{name}_final_images"

    csv_ok = csv_path.exists()
    folder_ok = image_dir.exists()

    if not csv_ok:
        print(f"{name:<10} {'NO':>7} {'-' if not folder_ok else 'YES':>10} {'-':>7}")
        continue

    df = pd.read_csv(csv_path, low_memory=False)
    rows = len(df)
    nan_text = df["text"].isna().sum()
    dup_url = df["article_url"].duplicated().sum()

    filenames_col = df["image_filename"].fillna("")
    uses_semicolon = filenames_col.str.contains(";").sum() > 0
    uses_pipe = filenames_col.str.contains(r"\|", regex=True).sum() > 0
    sep = ";" if uses_semicolon and not uses_pipe else ("|" if uses_pipe and not uses_semicolon else "MIXED/NONE")

    referenced = set()
    for filenames in filenames_col:
        for filename in re.split(r"[;|]", filenames):
            filename = filename.strip()
            if filename:
                referenced.add(filename)

    with_img = (filenames_col.str.strip() != "").sum()

    if folder_ok:
        on_disk = {f.name for f in image_dir.glob("*") if f.is_file()}
        missing_img = len(referenced - on_disk)
        orphan_files = len(on_disk - referenced)
    else:
        on_disk = set()
        missing_img = len(referenced)
        orphan_files = 0

    print(f"{name:<10} {'YES':>7} {'YES' if folder_ok else 'NO':>10} {rows:>7} {nan_text:>9} {dup_url:>8} {sep:>5} {with_img:>9} {missing_img:>12} {orphan_files:>13}")

In [ ]:
final_dir = Path(".")
df = pd.read_csv(final_dir / "spiegel_final_data.csv", low_memory=False)

image_count = df["image_filename"].fillna("").apply(lambda f: len([x for x in str(f).split(";") if x.strip()]))

print("distribution of image count per article:")
print(image_count.value_counts().sort_index())

print()
print("articles with more than 3 images:", (image_count > 3).sum())
print("articles with more than 4 images:", (image_count > 4).sum())
print("max images in a single article:", image_count.max())

In [ ]:
# remove spiegel articles with 19+ images (outliers - likely photo galleries/liveblogs)
import pandas as pd
from pathlib import Path

final_dir = Path(".")
image_dir = final_dir / "spiegel_final_images"

df = pd.read_csv(final_dir / "spiegel_final_data.csv", low_memory=False)
image_count = df["image_filename"].fillna("").apply(lambda f: len([x for x in str(f).split(";") if x.strip()]))

before = len(df)
removed = df[image_count >= 19].copy()
df = df[image_count < 19].copy()

print("removing", len(removed), "articles with 19+ images")
print(before, "->", len(df), "articles remaining")

# keep a record of what was excluded, for the methodology section
removed.to_csv(final_dir / "spiegel_removed_high_image_count.csv", index=False, encoding="utf-8-sig")
print("removed articles saved to:", final_dir / "spiegel_removed_high_image_count.csv")

In [ ]:
# sync spiegel_final_images: remove files no longer referenced by any remaining article
needed_filenames = set()
for filenames in df["image_filename"].fillna(""):
    for filename in str(filenames).split(";"):
        filename = filename.strip()
        if filename:
            needed_filenames.add(filename)

removed_files = 0
for existing_file in image_dir.glob("*"):
    if existing_file.name not in needed_filenames:
        existing_file.unlink()
        removed_files += 1

print(removed_files, "orphaned image files removed from", image_dir)
print(len(needed_filenames), "files still needed")

In [ ]:
# save the filtered spiegel dataset
df.to_csv(final_dir / "spiegel_final_data.csv", index=False, encoding="utf-8-sig")
print("saved:", final_dir / "spiegel_final_data.csv", "-", len(df), "articles")

## Delete krim=criminal

In [ ]:
# re-apply the corrected title filter to the final datasets and split off the articles it no longer matches
import pandas as pd
import re

patterns = {
    "de": r"\b(russland\w*|russisch\w*|kreml\w*|putin\w*|moskau\w*|"
          r"lawrow\w*|lavrov\w*|peskow\w*|peskov\w*|schoigu\w*|shoigu\w*|"
          r"gerassimow\w*|gerasimow\w*|medwedew\w*|sacharowa\w*|mischustin\w*|"
          r"gazprom\w*|rosneft\w*|rosatom\w*|transneft\w*|lukoil\w*|"
          r"donezk\w*|luhansk\w*|krim\b|krims\b|krimhalbinsel\w*|crimea\w*)\b",

    "pl": r"\b(rosj\w*|rosyjsk\w*|kreml\w*|putin\w*|moskw\w*|"
          r"ławrow\w*|lawrow\w*|pieskow\w*|szojgu\w*|szoigu\w*|shoigu\w*|"
          r"gierasimow\w*|miedwiediew\w*|zacharowa\w*|miszustin\w*|"
          r"gazprom\w*|rosnieft\w*|rosatom\w*|transnieft\w*|łukoil\w*|lukoil\w*|"
          r"donieck\w*|ługańsk\w*|luhansk\w*|krym(?:u|ie|em|owi)?\b|krymsk\w*)\b",

    "hu": r"\b(oroszorsz[aá]g\w*|orosz(?!l[aá]n)\w*|kreml\w*|putyin\w*|moszkv\w*|"
          r"lavrov\w*|peszkov\w*|peskov\w*|sojgu\w*|szojgu\w*|shoigu\w*|"
          r"geraszimov\w*|gerasimov\w*|medvegyev\w*|medvedev\w*|zaharova\w*|zakharova\w*|"
          r"misustyin\w*|mishustin\w*|"
          r"gazprom\w*|rosznyefty\w*|rosneft\w*|roszatom\w*|rosatom\w*|"
          r"transznyefty\w*|transneft\w*|lukoil\w*|"
          r"donyeck\w*|donetsk\w*|luhanszk\w*|krím\w*|krim\b)\b",

    "es": r"\b(rusia\w*|rus[oa]s?|kremlin\w*|putin\w*|mosc[uú]\w*|"
          r"lavrov\w*|peskov\w*|shoigu\w*|choigu\w*|gerasimov\w*|"
          r"medv[eé]dev\w*|zaj[aá]rova\w*|zakharova\w*|mishustin\w*|mishust[ií]n\w*|"
          r"gazprom\w*|rosneft\w*|rosatom\w*|transneft\w*|lukoil\w*|"
          r"donetsk\w*|lugansk\w*|crimea\w*)\b",
}

sources = {
    "spiegel": "de", "welt": "de",
    "eldiario": "es", "elmundo": "es",
    "polityka": "pl", "rp": "pl",
    "magyar": "hu", "telex": "hu",
}

for name, lang in sources.items():
    df = pd.read_csv(f"{name}_final_data.csv", low_memory=False)
    title = df["title"].fillna("").astype(str)
    keep = title.str.contains(patterns[lang], case=False, regex=True)

    removed = df[~keep]
    removed.to_csv(f"{name}_removed_by_corrected_filter.csv", index=False, encoding="utf-8-sig")
    df[keep].to_csv(f"{name}_final_data_v2.csv", index=False, encoding="utf-8-sig")

    print(name, len(df), "->", int(keep.sum()), "removed", len(removed))
    for t in removed["title"].head(10):
        print("   ", t)

In [ ]:
# re-run the corrected title filter for the german outlets only
import pandas as pd
import re

pattern_de = (
    r"\b(?:russland\w*|russisch\w*|kreml\w*|putin\w*|moskau\w*|"
    r"lawrow\w*|lavrov\w*|peskow\w*|peskov\w*|schoigu\w*|shoigu\w*|"
    r"gerassimow\w*|gerasimow\w*|medwedew\w*|sacharowa\w*|mischustin\w*|"
    r"gazprom\w*|rosneft\w*|rosatom\w*|transneft\w*|lukoil\w*|"
    r"donezk\w*|luhansk\w*|krim(?!i)\w*|crimea\w*)\b"
)

for name in ["spiegel", "welt"]:
    df = pd.read_csv(f"{name}_final_data.csv", low_memory=False)
    title = df["title"].fillna("").astype(str)
    keep = title.str.contains(pattern_de, case=False, regex=True)

    df[~keep].to_csv(f"{name}_removed_by_corrected_filter.csv", index=False, encoding="utf-8-sig")
    df[keep].to_csv(f"{name}_final_data_v2.csv", index=False, encoding="utf-8-sig")

    print(name, len(df), "->", int(keep.sum()), "removed", len(df) - int(keep.sum()))
    for t in df.loc[~keep, "title"].head(15):
        print("   ", t)

In [ ]:
# rebuild the final corpora with the corrected title filter
import pandas as pd
from pathlib import Path

rp_file = Path(
    "."
    r"\Desktop\Article 2\poland_data\RP Poland\rp_final_data_with_dates.csv"
)

patterns = {
    "de": r"\b(?:russland\w*|russisch\w*|kreml\w*|putin\w*|moskau\w*|"
          r"lawrow\w*|lavrov\w*|peskow\w*|peskov\w*|schoigu\w*|shoigu\w*|"
          r"gerassimow\w*|gerasimow\w*|medwedew\w*|sacharowa\w*|mischustin\w*|"
          r"gazprom\w*|rosneft\w*|rosatom\w*|transneft\w*|lukoil\w*|"
          r"donezk\w*|luhansk\w*|krim(?!i)\w*|crimea\w*)\b",

    "es": r"\b(?:rusia\w*|rus[oa]s?|kremlin\w*|putin\w*|mosc[uú]\w*|"
          r"lavrov\w*|peskov\w*|shoigu\w*|choigu\w*|gerasimov\w*|"
          r"medv[eé]dev\w*|zaj[aá]rova\w*|zakharova\w*|mishustin\w*|mishust[ií]n\w*|"
          r"gazprom\w*|rosneft\w*|rosatom\w*|transneft\w*|lukoil\w*|"
          r"donetsk\w*|lugansk\w*|crimea\w*)\b",

    "pl": r"\b(?:rosj\w*|rosyjsk\w*|kreml\w*|putin\w*|moskw\w*|"
          r"ławrow\w*|lawrow\w*|pieskow\w*|szojgu\w*|szoigu\w*|shoigu\w*|"
          r"gierasimow\w*|miedwiediew\w*|zacharowa\w*|miszustin\w*|"
          r"gazprom\w*|rosnieft\w*|rosatom\w*|transnieft\w*|łukoil\w*|lukoil\w*|"
          r"donieck\w*|ługańsk\w*|luhansk\w*|krym(?:u|ie|em|owi)?\b|krymsk\w*)\b",

    "hu": r"\b(?:oroszorsz[aá]g\w*|orosz(?!l[aá]n)\w*|kreml\w*|putyin\w*|moszkv\w*|"
          r"lavrov\w*|peszkov\w*|peskov\w*|sojgu\w*|szojgu\w*|shoigu\w*|"
          r"geraszimov\w*|gerasimov\w*|medvegyev\w*|medvedev\w*|zaharova\w*|zakharova\w*|"
          r"misustyin\w*|mishustin\w*|"
          r"gazprom\w*|rosznyefty\w*|rosneft\w*|roszatom\w*|rosatom\w*|"
          r"transznyefty\w*|transneft\w*|lukoil\w*|"
          r"donyeck\w*|donetsk\w*|luhanszk\w*|krím\w*|krim\b)\b",
}

sources = {
    "spiegel":  ("de", "Germany", "spiegel_final_data.csv"),
    "welt":     ("de", "Germany", "welt_final_data.csv"),
    "eldiario": ("es", "Spain",   "eldiario_final_data.csv"),
    "elmundo":  ("es", "Spain",   "elmundo_final_data.csv"),
    "polityka": ("pl", "Poland",  "polityka_final_data.csv"),
    "rp":       ("pl", "Poland",  rp_file),
    "magyar":   ("hu", "Hungary", "magyar_final_data.csv"),
    "telex":    ("hu", "Hungary", "telex_final_data.csv"),
}

rows = []

for name, (lang, country, corpus_file) in sources.items():
    df = pd.read_csv(corpus_file, low_memory=False)
    title = df["title"].fillna("").astype(str)
    keep = title.str.contains(patterns[lang], case=False, regex=True)

    corpus = df[keep]
    corpus.to_csv(f"{name}_final_data_v2.csv", index=False, encoding="utf-8-sig")

    rows.append({
        "outlet": name,
        "country": country,
        "before": len(df),
        "removed": int((~keep).sum()),
        "articles": len(corpus),
    })

summary = pd.DataFrame(rows)
summary.to_csv("corpus_summary_v2.csv", index=False, encoding="utf-8-sig")

print(summary.to_string(index=False))
print()
print(summary.groupby("country")["articles"].sum().to_string())
print()
print("total articles:", summary["articles"].sum())